# Fixed Rate of Inconclusive Outcome

In [1]:
import sys
sys.path.append("../")

In [2]:
# Set up logging to display in Jupyter Notebook
import logging

logging.basicConfig(
    level=logging.ERROR,  # Show ERROR and above
    format="%(levelname)s: %(message)s",  # Format: "ERROR: message"
    force=True,  # Needed in Jupyter to reset config
)

In [3]:
from flow.solve_mix import *
from flow.interface import *
from utils.handy_states import *
from utils.holevo_bound import *

In [ ]:
state_dict = simple_2(0.2, 0.5, 0.7)
num_qubits = state_dict["num_qubits"]
num_states = state_dict["num_states"]
state_vec = state_dict["state_vec"]
dense_mat = [DensityMatrix(_) for _ in state_vec]

In [5]:
#'''
num_qubits = 1
num_states = 2

state_vec = [
    Statevector([1, 0]),
    Statevector([1/np.sqrt(2), 1/np.sqrt(2)]),
]
dense_mat = [DensityMatrix(_) for _ in state_vec]
#'''


In [6]:
num_states

2

In [7]:
# Define QSD Problem
qsd_problem = ProblemSpec(
    num_qubits=num_qubits,
    num_states=num_states,
    case_id="0813_test",
    state_type="densitymatrix",
)

In [8]:
n = num_states
# Comment/uncomment to switch between different prior probabilities
prior_prob = np.ones(n) * (1 / n)
# prior_prob = [1/9, 3/9, 5/9]
qsd_problem.prior_prob = prior_prob

In [9]:
# Compute Holevo bound for different dep_noise values
for dep_noise in [0.0, 0.001, 0.1]:
    dense_mat_data = [dense_mat[i].data for i in range(len(dense_mat))]
    chi = holevo_bound(dense_mat_data, prior_prob, dep_noise)
    print(f"Holevo bound (dep_noise={dep_noise}): {chi:.4f} bits")

Holevo bound (dep_noise=0.0): 0.6009 bits
Holevo bound (dep_noise=0.001): 0.5956 bits
Holevo bound (dep_noise=0.1): 0.3976 bits


In [10]:
noise_levels = [0] + [0.1 ** (6 - 0.5 * i) for i in range(11)]
disturbance_states = [
    DensityMatrix(
        ProblemSpec.depolarizing_noise_channel(num_qubits=num_qubits)
    )
    for _ in range(num_states)
]

In [11]:
qsd_problem.set_states(
    state_type="densitymatrix",
    states=dense_mat,
    overwrite=True,
)

In [12]:
cvxpy_problem = frio_problem(
    problem_spec=qsd_problem,
    p_inc_lb=0.3,
    prior_prob=prior_prob,
)

In [13]:
def run_qsd(
    qsd_problem: ProblemSpec,
    cvxpy_problem,
    p_succ_result,
    p_err_result,
    p_inc_result,
    eps=1e-8,
):
    cvxpy_settings = {"solver": cp.MOSEK, "verbose": False, "eps": eps}
    cvxpy_problem.solve(**cvxpy_settings)

    # print(cvxpy_problem.status)
    # print(cvxpy_problem.solution.opt_val)
    vars = cvxpy_problem.variables()

    frio_povm = [var.value for var in vars]
    # print(frio_povm)

    prob_mat = calculate_prob_matrix_simple(
        prior_probs=prior_prob,
        povm=frio_povm,
        states=qsd_problem.states,
    )

    k = qsd_problem.num_states
    p_succ = 0
    for i in range(qsd_problem.num_states):
        p_succ += prob_mat[i][i]
    p_succ_result.append(p_succ)

    p_err = 0
    for i in range(qsd_problem.num_states):
        for j in range(qsd_problem.num_states):
            if i != j:
                p_err += prob_mat[i][j]
    p_err_result.append(p_err)

    p_inc = 0
    for i in range(k):
        p_inc += prob_mat[i][k]
    p_inc_result.append(p_inc)

    mi = mutual_information(
        prob_mat=prob_mat,
        prior_prob=prior_prob,
        k=num_states,
    )

    print(
        np.format_float_scientific(p_succ, 4),
        np.format_float_scientific(p_err, 4),
        np.format_float_scientific(p_inc, 4),
        np.format_float_scientific(p_succ + p_err + p_inc, 4),
        np.format_float_scientific(10 * np.log10(p_succ / p_err), 4),
        np.format_float_scientific(mi, 4),
    )

    # print("alpha", np.array(calculate_errors(prob_mat)[0]))
    # print("beta ", max(calculate_errors(prob_mat)[1]))
    #
    # print()

In [14]:
params = [0.1 * i for i in range(10)]
p_succ_result = []
p_err_result = []
p_inc_result = []
print("p_succ\tp_err\tp_inc\tsum\tpSNR\tMutual information")
noise_levels = [0]
for param in params:
    print("lb =", param)
    print()
    for noise_level in noise_levels:
        print("dep noise =", noise_level)
        combined_states = [
            (1 - noise_level) * dense_mat[_]
            + noise_level * disturbance_states[_].data
            for _ in range(num_states)
        ]
        qsd_problem.set_states(
            state_type="densitymatrix",
            states=combined_states,
            overwrite=True,
        )
        cvxpy_problem = frio_problem(
            problem_spec=qsd_problem,
            p_inc_lb=param,
            prior_prob=prior_prob,
        )
        run_qsd(
            qsd_problem=qsd_problem,
            cvxpy_problem=cvxpy_problem,
            p_succ_result=p_succ_result,
            p_err_result=p_err_result,
            p_inc_result=p_inc_result,
            eps=1e-8,
        )
        print()

p_succ	p_err	p_inc	sum	pSNR	Mutual information
lb = 0.0

dep noise = 0
8.5355e-01 1.4645e-01 3.2931e-09 1.e+00 7.6555e+00 3.9912e-01

lb = 0.1

dep noise = 0
7.822e-01 1.1780e-01 1.0000e-01 1.e+00 8.2216e+00 3.9611e-01

lb = 0.2

dep noise = 0
7.0937e-01 9.0629e-02 2.e-01 1.e+00 8.9361e+00 3.9220e-01

lb = 0.30000000000000004

dep noise = 0
6.3472e-01 6.5279e-02 3.e-01 1.e+00 9.8781e+00 3.8693e-01

lb = 0.4

dep noise = 0


/home/ChienKaiMa/QSD/.venv/lib/python3.12/site-packages/mosek/__init__.py:18617: UserWarning: Argument sub in putvarboundlist: Incorrect array format causing data to be copied
  warnings.warn("Argument sub in putvarboundlist: Incorrect array format causing data to be copied");
/home/ChienKaiMa/QSD/.venv/lib/python3.12/site-packages/mosek/__init__.py:18925: UserWarning: Argument subj in putclist: Incorrect array format causing data to be copied
  warnings.warn("Argument subj in putclist: Incorrect array format causing data to be copied");
/home/ChienKaiMa/QSD/.venv/lib/python3.12/site-packages/mosek/__init__.py:18349: UserWarning: Argument sub in putconboundlist: Incorrect array format causing data to be copied
  warnings.warn("Argument sub in putconboundlist: Incorrect array format causing data to be copied");


5.5772e-01 4.2277e-02 4.e-01 1.e+00 1.1203e+01 3.7942e-01

lb = 0.5

dep noise = 0
4.7754e-01 2.2455e-02 5.e-01 1.e+00 1.3277e+01 3.6782e-01

lb = 0.6000000000000001

dep noise = 0
3.927e-01 7.3033e-03 6.e-01 1.e+00 1.7305e+01 3.4738e-01

lb = 0.7000000000000001

dep noise = 0
2.9996e-01 4.2097e-05 7.e-01 1.e+00 3.8528e+01 2.9940e-01

lb = 0.8

dep noise = 0
2.e-01 3.3970e-09 8.e-01 1.e+00 7.7699e+01 2.0000e-01

lb = 0.9

dep noise = 0
1.e-01 1.1193e-09 9.e-01 1.e+00 7.9511e+01 1.0000e-01



In [15]:
params = [0.1 * i for i in range(10)]
p_succ_result = []
p_err_result = []
p_inc_result = []
print("p_succ\tp_err\tp_inc\tpSNR\tMutual information\tsum")
noise_levels = [1e-3]
for param in params:
    print("lb =", param)
    print()
    for noise_level in noise_levels:
        print("dep noise =", noise_level)
        combined_states = [
            (1 - noise_level) * dense_mat[_]
            + noise_level * disturbance_states[_].data
            for _ in range(num_states)
        ]
        qsd_problem.set_states(
            state_type="densitymatrix",
            states=combined_states,
            overwrite=True,
        )
        cvxpy_problem = frio_problem(
            problem_spec=qsd_problem,
            p_inc_lb=param,
            prior_prob=prior_prob,
        )
        run_qsd(
            qsd_problem=qsd_problem,
            cvxpy_problem=cvxpy_problem,
            p_succ_result=p_succ_result,
            p_err_result=p_err_result,
            p_inc_result=p_inc_result,
            eps=1e-8,
        )
        print()

p_succ	p_err	p_inc	pSNR	Mutual information	sum
lb = 0.0

dep noise = 0.001
8.532e-01 1.4680e-01 3.2969e-09 1.e+00 7.6432e+00 3.9823e-01

lb = 0.1

dep noise = 0.001
7.8186e-01 1.1814e-01 1.0000e-01 1.e+00 8.2072e+00 3.9518e-01

lb = 0.2

dep noise = 0.001
7.0904e-01 9.0958e-02 2.e-01 1.e+00 8.9183e+00 3.9123e-01

lb = 0.30000000000000004

dep noise = 0.001
6.3440e-01 6.5596e-02 3.e-01 1.e+00 9.8549e+00 3.8589e-01

lb = 0.4

dep noise = 0.001
5.5742e-01 4.2581e-02 4.e-01 1.e+00 1.117e+01 3.7828e-01

lb = 0.5

dep noise = 0.001
4.7725e-01 2.2749e-02 5.e-01 1.e+00 1.3218e+01 3.6652e-01

lb = 0.6000000000000001

dep noise = 0.001
3.9241e-01 7.5904e-03 6.e-01 1.e+00 1.7135e+01 3.4574e-01

lb = 0.7000000000000001

dep noise = 0.001
2.9967e-01 3.3362e-04 7.e-01 1.e+00 2.9534e+01 2.9625e-01

lb = 0.8

dep noise = 0.001
1.9980e-01 1.9970e-04 8.e-01 1.e+00 3.0002e+01 1.9772e-01

lb = 0.9

dep noise = 0.001
9.9900e-02 9.9851e-05 9.e-01 1.e+00 3.0002e+01 9.8861e-02



In [16]:
params = [0.1 * i for i in range(10)]
p_succ_result = []
p_err_result = []
p_inc_result = []
print("p_succ\tp_err\tp_inc\tpSNR\tMutual information\tsum")
noise_levels = [1e-1]
for param in params:
    print("lb =", param)
    print()
    for noise_level in noise_levels:
        print("dep noise =", noise_level)
        combined_states = [
            (1 - noise_level) * dense_mat[_]
            + noise_level * disturbance_states[_].data
            for _ in range(num_states)
        ]
        qsd_problem.set_states(
            state_type="densitymatrix",
            states=combined_states,
            overwrite=True,
        )
        cvxpy_problem = frio_problem(
            problem_spec=qsd_problem,
            p_inc_lb=param,
            prior_prob=prior_prob,
        )
        run_qsd(
            qsd_problem=qsd_problem,
            cvxpy_problem=cvxpy_problem,
            p_succ_result=p_succ_result,
            p_err_result=p_err_result,
            p_inc_result=p_inc_result,
            eps=1e-8,
        )
        print()

p_succ	p_err	p_inc	pSNR	Mutual information	sum
lb = 0.0

dep noise = 0.1
8.182e-01 1.8180e-01 3.569e-09 1.e+00 6.5326e+00 3.16e-01

lb = 0.1

dep noise = 0.1
7.4812e-01 1.5188e-01 1.0000e-01 1.e+00 6.9247e+00 3.1063e-01

lb = 0.2

dep noise = 0.1
6.7659e-01 1.2341e-01 2.e-01 1.e+00 7.3896e+00 3.0367e-01

lb = 0.30000000000000004

dep noise = 0.1
6.0323e-01 9.6769e-02 3.e-01 1.e+00 7.9475e+00 2.9427e-01

lb = 0.4

dep noise = 0.1
5.2749e-01 7.2512e-02 4.e-01 1.e+00 8.6181e+00 2.8092e-01

lb = 0.5

dep noise = 0.1
4.4843e-01 5.1566e-02 5.e-01 1.e+00 9.3934e+00 2.6058e-01

lb = 0.6000000000000001

dep noise = 0.1
3.6432e-01 3.5679e-02 6.e-01 1.e+00 1.0091e+01 2.2649e-01

lb = 0.7000000000000001

dep noise = 0.1
2.7375e-01 2.6246e-02 7.e-01 1.e+00 1.0183e+01 1.7173e-01

lb = 0.8

dep noise = 0.1
1.8250e-01 1.7497e-02 8.e-01 1.e+00 1.0183e+01 1.144e-01

lb = 0.9

dep noise = 0.1
9.1251e-02 8.7486e-03 9.e-01 1.e+00 1.0183e+01 5.7198e-02

